# Syntactic pipeline

Now moving onto the pipeline for syntactic features. Previously used NLTK but now must utilise Stanza for parsing.

In [1]:
import pandas as pd
import numpy as np
import re
import stanza
from collections import Counter
from nltk import Tree

Create stanza pipline for later.

In [13]:
stan_pipe = stanza.Pipeline(
    lang = "en",
    # need these processors for feature extraction later
    # don't actually need lemmatisation but that is needed for depparse
    processors = "tokenize, pos, lemma, depparse, constituency",
    use_gpu = False,
    verbose = False
)

Now need a cleaning function. Copying this over from Surface pipeline.

In [14]:
def syntactic_pipe_clean(text):

    # if text is missing return a blank ""
    if pd.isna(text):
        return ""

    # convert text to string if not already
    text = str(text)

    # remove any html markup still existing
    text = re.sub(r"<[^>]+>", " ", text)

    # change any multiple spaces, tabs, newlines with " "
    text = re.sub(r"\s+", " ", text)

    # strip outside whitespace
    text = text.strip()

    # return cleaned text
    return text

Will put a function here to call from within pipeline to calculate dependency tree depth.

In [15]:
def get_dep_depth(word_id, paths):

    # initialise depth = 0
    depth = 0
    # and begin with current word
    current = word_id

    while (
        # check current word exists in dict of heads
        current in paths
        # and continue until head of current word = 0, i.e. root
        and paths[current] != 0
    ):
        # every iteration increment depth
        depth += 1
        # and move up a level to head of current word
        current = paths[current]

    # return depth of current word
    return depth

________

# NOTE

Went through multiple iterations of pipline due to ammendments. Have removed those early piplines as I assume you only want to see what is necessary but have left som results in case you are interested. 

__________________

BLANK

Pipeline should be working now, lets check on a piece of text. Following text same as the one used in surface pipeline.

In [6]:
test_text = """

Conservative leader Kemi Badenoch has pressed Andy Burnham over UK borrowing and spending at his first Prime Minister's Questions.
Badenoch called on the PM to say where spending would be cut and how he would deal with the UK's increasing debt, after the cost of 
borrowing for the UK rose, reaching a new 18-year high on Wednesday. The PM blamed the previous Conservative government saying that the 
"turbulence on global markets are because of that exposure that they left behind". Burnham became prime minister in July, having taken 
over from Sir Keir Starmer just before Parliament's summer recess, and it was the first time he had faced the Tory leader at the Commons session.
The prime minister used the summer recess to tour the UK, and had announced a series of eye-catching policies, mainly aimed at tackling the 
cost of living. MPs returned to Parliament on Tuesday and Burnham spent more than three hours in the Commons, making a statement and 
answering questions from MPs. At PMQs, Badenoch said the prime minister had set out plenty of spending plans but no detail of how he would be 
paying the bill. "The markets are clearly worried that we now have a spendthrift prime minister who wants to say yes to everyone but cannot 
tell us where the money is coming from," she said. "He needs to pay for all his new promises and he has a choice - higher taxes, more borrowing, 
or spending cuts. "Yesterday he said change begins with honesty, so will he be honest and tell us: is he preparing to raise taxes again, yes or no?"
Burnham had earlier said he had already cut taxes, including VAT on energy bills and a cut in business rates for hospitality. "I've already 
indicated my first moves were to cut tax, but I'm not going to do what any prime minister has done and write the budget here," he said, adding 
he and Chancellor John Healey had agreed on an early date for the Budget in October to reduce speculation. At PMQs, Badenoch also raised comments 
from economist and cross-bench peer Lord Jim O'Neill, who had been tipped to become Burnham's chief economic adviser, but who ruled out joining 
Burnham's government. Speaking to the BBC, Lord O'Neill had praised the PM for "a great first five weeks" which he said had helped lift consumer 
and business confidence; adding the current spike in market interest rates for government debt would force Labour to start "dealing with the triple 
lock" on the state pension, and "excessive" welfare spending. Badenoch pointed to a warning from Lord O'Neill that the prime minister's tone 
during his Tuesday Commons debut would have unsettled investors, although the peer had also said that spike was mostly mirroring US market 
upheaval concerning the Iran conflict.

"""

In [7]:
# run pipeline on test text
test_features = extract_syntactic_features(test_text)
# print out values
for feature, value in test_features.items():
    print(f"{feature}: {value}")

mean_parse_tree_depth: 15.733333333333333
mean_clausal_dep_per_sent: 3.1333333333333333
sent_with_clausal_dep_ratio: 0.9333333333333333
mean_prod_rules_per_sent: 31.866666666666667
mean_dep_depth: 3.236043207584415
mean_dep_distance: 3.894230769230769
pos_ratio_ADJ: 0.05420560747663551
pos_ratio_NOUN: 0.1925233644859813
pos_ratio_PROPN: 0.09158878504672897
pos_ratio_AUX: 0.06355140186915888
pos_ratio_VERB: 0.13457943925233645
pos_ratio_ADP: 0.08971962616822429
pos_ratio_CCONJ: 0.037383177570093455
pos_ratio_PRON: 0.05794392523364486
pos_ratio_PART: 0.03364485981308411
pos_ratio_PUNCT: 0.10841121495327102
pos_ratio_DET: 0.07850467289719626
pos_ratio_ADV: 0.03364485981308411
pos_ratio_SCONJ: 0.013084112149532711
pos_ratio_NUM: 0.005607476635514018
pos_ratio_INTJ: 0.005607476635514018
pos_bigram_ADJ_NOUN: 0.03461538461538462
pos_bigram_NOUN_PROPN: 0.0038461538461538464
pos_bigram_PROPN_PROPN: 0.021153846153846155
pos_bigram_PROPN_AUX: 0.007692307692307693
pos_bigram_AUX_VERB: 0.0307692307

Removed trigrams as there was too many sparse features. Bigrams suffers from a similar issue but we do not want to remove them as they provide useful info, so now the plan is to get all bigrams from the training set only and have a look at the most frequent and then ammend the pipeline to only consider these most frequent.

NB: This was a long bigram analysis across corpus cell and output but again assume you only want results.

In [11]:
bigram_results_df.to_csv("pos_bigram_analysis.csv", index = False)

___

In [2]:
data = pd.read_csv("pos_bigram_analysis.csv")

In [7]:
data.head(20)

,bigram,pos_1,pos_2,corpus_frequency,text_frequency,text_frequency_ratio
0,ADJ_NOUN,ADJ,NOUN,64342,3199,0.999687
1,NOUN_PUNCT,NOUN,PUNCT,89069,3198,0.999375
2,DET_NOUN,DET,NOUN,79788,3198,0.999375
3,NOUN_ADP,NOUN,ADP,54002,3198,0.999375
4,ADP_DET,ADP,DET,50775,3198,0.999375
5,VERB_ADP,VERB,ADP,36397,3198,0.999375
6,DET_ADJ,DET,ADJ,32762,3197,0.999062
7,AUX_VERB,AUX,VERB,32278,3197,0.999062
8,VERB_DET,VERB,DET,30576,3196,0.998750
9,ADP_NOUN,ADP,NOUN,26231,3188,0.996250


Hard to find an obvious cut-off point there.

In [8]:
# set bin boundaries
boundaries = [0.99, 0.95, 0.9, 0.8, 0.7, 0.6, 0.5]
#initialise empty list
results = []

# loop through the set bins above
for boundary in boundaries:

    # get num of text_freq-Ratios above that boundary
    bigram_count = (data["text_frequency_ratio"] >= boundary).sum()

    # add results to new DF
    results.append ({

        "min_freq_ratio" : boundary,
        "min_percent_of_texts" : boundary * 100,
        "num_bigrams" : bigram_count

    })

results = pd.DataFrame(results)
results

,min_freq_ratio,min_percent_of_texts,num_bigrams
0,0.99,99.0,10
1,0.95,95.0,21
2,0.90,90.0,34
3,0.80,80.0,49
4,0.70,70.0,65
5,0.60,60.0,76
6,0.50,50.0,87


Nothing too concrete. I had hoped to use a max of 20-30 so choosing the top 34 bigrams that appear in 90% of texts across the training set seems a good stopping point.

In [16]:
# go through total list of bigrams and get those whose ratio > 0.9
# i.e. appear in >= 90% of texts
bigram_list = list(
    data.loc[
        data["text_frequency_ratio"] >= 0.9,
        ["pos_1", "pos_2"]
    ].itertuples(index = False, name = None)
)
print("Bigrams chosen: ", len(bigram_list))

Bigrams chosen:  34


In [17]:
bigram_list

[('ADJ', 'NOUN'),
 ('NOUN', 'PUNCT'),
 ('DET', 'NOUN'),
 ('NOUN', 'ADP'),
 ('ADP', 'DET'),
 ('VERB', 'ADP'),
 ('DET', 'ADJ'),
 ('AUX', 'VERB'),
 ('VERB', 'DET'),
 ('ADP', 'NOUN'),
 ('PRON', 'VERB'),
 ('NOUN', 'AUX'),
 ('PART', 'VERB'),
 ('NOUN', 'VERB'),
 ('VERB', 'NOUN'),
 ('PRON', 'AUX'),
 ('ADV', 'VERB'),
 ('NOUN', 'NOUN'),
 ('NOUN', 'CCONJ'),
 ('PUNCT', 'CCONJ'),
 ('VERB', 'PUNCT'),
 ('ADP', 'ADJ'),
 ('VERB', 'PRON'),
 ('ADJ', 'PUNCT'),
 ('AUX', 'ADV'),
 ('PUNCT', 'PRON'),
 ('VERB', 'ADV'),
 ('ADP', 'PRON'),
 ('CCONJ', 'VERB'),
 ('PUNCT', 'VERB'),
 ('PRON', 'NOUN'),
 ('NOUN', 'ADV'),
 ('ADJ', 'ADP'),
 ('VERB', 'ADJ')]

In [18]:
chosen_bigrams_df = pd.DataFrame(bigram_list, columns = ["pos_1", "pos_2"])
chosen_bigrams_df.to_csv("final_chosen_bigrams.csv", index = False)

In [19]:
chosen_bigrams_df

,pos_1,pos_2
0,ADJ,NOUN
1,NOUN,PUNCT
2,DET,NOUN
3,NOUN,ADP
4,ADP,DET
5,VERB,ADP
6,DET,ADJ
7,AUX,VERB
8,VERB,DET
9,ADP,NOUN


___

Ok now we have out final list of bigrams. I will copy paste the main pipeline from above and upate it with this list so that the final version of the pipeline is executed from here.

In [14]:
bigrams = pd.read_csv("final_chosen_bigrams.csv")

In [15]:
chosen_bigrams_list = list(
    bigrams[["pos_1", "pos_2"]]
    .itertuples(index = False, name = None)
)

print(chosen_bigrams_list)

[('ADJ', 'NOUN'), ('NOUN', 'PUNCT'), ('DET', 'NOUN'), ('NOUN', 'ADP'), ('ADP', 'DET'), ('VERB', 'ADP'), ('DET', 'ADJ'), ('AUX', 'VERB'), ('VERB', 'DET'), ('ADP', 'NOUN'), ('PRON', 'VERB'), ('NOUN', 'AUX'), ('PART', 'VERB'), ('NOUN', 'VERB'), ('VERB', 'NOUN'), ('PRON', 'AUX'), ('ADV', 'VERB'), ('NOUN', 'NOUN'), ('NOUN', 'CCONJ'), ('PUNCT', 'CCONJ'), ('VERB', 'PUNCT'), ('ADP', 'ADJ'), ('VERB', 'PRON'), ('ADJ', 'PUNCT'), ('AUX', 'ADV'), ('PUNCT', 'PRON'), ('VERB', 'ADV'), ('ADP', 'PRON'), ('CCONJ', 'VERB'), ('PUNCT', 'VERB'), ('PRON', 'NOUN'), ('NOUN', 'ADV'), ('ADJ', 'ADP'), ('VERB', 'ADJ')]


Cell below is now a copy of the main pipeline above, with chosen bigrams added in and other parts such as trigrams fully removed.

___________

NB: THis was the second pipline which isn't final so also removed but left discussion of changes after.

In [17]:
test_text = """

Conservative leader Kemi Badenoch has pressed Andy Burnham over UK borrowing and spending at his first Prime Minister's Questions.
Badenoch called on the PM to say where spending would be cut and how he would deal with the UK's increasing debt, after the cost of 
borrowing for the UK rose, reaching a new 18-year high on Wednesday. The PM blamed the previous Conservative government saying that the 
"turbulence on global markets are because of that exposure that they left behind". Burnham became prime minister in July, having taken 
over from Sir Keir Starmer just before Parliament's summer recess, and it was the first time he had faced the Tory leader at the Commons session.
The prime minister used the summer recess to tour the UK, and had announced a series of eye-catching policies, mainly aimed at tackling the 
cost of living. MPs returned to Parliament on Tuesday and Burnham spent more than three hours in the Commons, making a statement and 
answering questions from MPs. At PMQs, Badenoch said the prime minister had set out plenty of spending plans but no detail of how he would be 
paying the bill. "The markets are clearly worried that we now have a spendthrift prime minister who wants to say yes to everyone but cannot 
tell us where the money is coming from," she said. "He needs to pay for all his new promises and he has a choice - higher taxes, more borrowing, 
or spending cuts. "Yesterday he said change begins with honesty, so will he be honest and tell us: is he preparing to raise taxes again, yes or no?"
Burnham had earlier said he had already cut taxes, including VAT on energy bills and a cut in business rates for hospitality. "I've already 
indicated my first moves were to cut tax, but I'm not going to do what any prime minister has done and write the budget here," he said, adding 
he and Chancellor John Healey had agreed on an early date for the Budget in October to reduce speculation. At PMQs, Badenoch also raised comments 
from economist and cross-bench peer Lord Jim O'Neill, who had been tipped to become Burnham's chief economic adviser, but who ruled out joining 
Burnham's government. Speaking to the BBC, Lord O'Neill had praised the PM for "a great first five weeks" which he said had helped lift consumer 
and business confidence; adding the current spike in market interest rates for government debt would force Labour to start "dealing with the triple 
lock" on the state pension, and "excessive" welfare spending. Badenoch pointed to a warning from Lord O'Neill that the prime minister's tone 
during his Tuesday Commons debut would have unsettled investors, although the peer had also said that spike was mostly mirroring US market 
upheaval concerning the Iran conflict.

"""

In [18]:
# run pipeline on test text
test_features = extract_syntactic_features_2(test_text)
# print out values
for feature, value in test_features.items():
    print(f"{feature}: {value}")

mean_parse_tree_depth: 15.733333333333333
mean_clausal_dep_per_sent: 3.1333333333333333
sent_with_clausal_dep_ratio: 0.9333333333333333
mean_prod_rules_per_sent: 31.866666666666667
mean_dep_depth: 3.236043207584415
mean_dep_distance: 3.894230769230769
pos_ratio_ADJ: 0.05420560747663551
pos_ratio_NOUN: 0.1925233644859813
pos_ratio_PROPN: 0.09158878504672897
pos_ratio_AUX: 0.06355140186915888
pos_ratio_VERB: 0.13457943925233645
pos_ratio_ADP: 0.08971962616822429
pos_ratio_CCONJ: 0.037383177570093455
pos_ratio_PRON: 0.05794392523364486
pos_ratio_PART: 0.03364485981308411
pos_ratio_PUNCT: 0.10841121495327102
pos_ratio_DET: 0.07850467289719626
pos_ratio_ADV: 0.03364485981308411
pos_ratio_SCONJ: 0.013084112149532711
pos_ratio_NUM: 0.005607476635514018
pos_ratio_INTJ: 0.005607476635514018
pos_bigram_ADJ_NOUN: 0.057692307692307696
pos_bigram_NOUN_PUNCT: 0.08333333333333333
pos_bigram_DET_NOUN: 0.0673076923076923
pos_bigram_NOUN_ADP: 0.07692307692307693
pos_bigram_ADP_DET: 0.041666666666666664


Now happy with output. A manageable amount of features, POS bigrams limited to those we selected. Considering this pipeline now complete.

________

# MAIN EXTRACTION

Now time to run this across all texts, but first to test on first 20. All code below now copied from Surface extraction section.

In [19]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run syntactic feature extraction on first 20 texts to ensure it works as expected
X_train_syntactic = pd.DataFrame(
    train_df["text"].iloc[:20].apply(extract_syntactic_features_2
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_syntactic.shape)
print(X_train_syntactic.isna().sum().sum())
print(X_train_syntactic.dtypes.value_counts())
X_train_syntactic.head(10)

(20, 95)
208
float64    95
Name: count, dtype: int64


,mean_parse_tree_depth,mean_clausal_dep_per_sent,sent_with_clausal_dep_ratio,mean_prod_rules_per_sent,mean_dep_depth,mean_dep_distance,pos_ratio_DET,pos_ratio_NUM,pos_ratio_PROPN,pos_ratio_NOUN,...,dependency_ratio_expl,dependency_ratio_csubj,pos_ratio_SYM,pos_ratio_INTJ,dependency_ratio_discourse,dependency_ratio_vocative,pos_ratio_X,dependency_ratio_list,dependency_ratio_dislocated,dependency_ratio_reparandum
0,10.954545,1.045455,0.681818,16.863636,2.292963,3.173594,0.069606,0.023202,0.058005,0.248260,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,13.750000,1.687500,0.750000,23.687500,2.426383,3.772840,0.099762,0.002375,0.009501,0.190024,...,0.004751,0.002375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,13.888889,1.555556,0.833333,21.555556,2.688738,3.200935,0.085202,0.024664,0.042601,0.246637,...,NaN,NaN,0.002242,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,13.842105,2.052632,0.947368,22.105263,2.626988,3.576674,0.103734,0.008299,0.053942,0.201245,...,0.004149,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10.370370,1.111111,0.481481,15.666667,1.895112,3.309577,0.086134,0.010504,0.018908,0.155462,...,NaN,NaN,0.008403,0.002101,0.002101,NaN,NaN,NaN,NaN,NaN
5,12.882353,1.235294,0.705882,22.529412,2.527591,3.378076,0.092672,0.043103,0.209052,0.150862,...,NaN,NaN,0.002155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,10.666667,1.428571,0.619048,21.380952,2.214483,3.738574,0.084507,0.007042,0.028169,0.132042,...,0.003521,NaN,NaN,0.005282,0.003521,0.003521,NaN,NaN,NaN,NaN
7,12.000000,1.894737,0.789474,20.842105,2.145926,3.502488,0.099762,0.002375,0.002375,0.149644,...,0.004751,0.002375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,12.722222,1.222222,0.555556,22.333333,2.679917,4.056911,0.056863,0.027451,0.039216,0.252941,...,0.003922,NaN,0.005882,NaN,0.005882,NaN,0.001961,0.009804,NaN,NaN
9,9.487179,0.948718,0.564103,12.512821,1.759073,2.877049,0.110057,0.007590,0.030361,0.161290,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Quite a lot of NaNs, but semi-expected as these are ratios, and so can be zero, but lets inspect where they're happening

In [20]:
nans = X_train_syntactic.isna().sum()
nans[nans > 0]

pos_ratio_PROPN                 1
pos_ratio_SCONJ                 1
phrase_per_sent_ADJP            1
dependency_ratio_flat           8
dependency_ratio_appos          7
dependency_ratio_nummod         2
dependency_ratio_fixed          1
dependency_ratio_ccomp          7
dependency_ratio_iobj          12
dependency_ratio_parataxis      7
dependency_ratio_expl           7
dependency_ratio_csubj         15
pos_ratio_SYM                  14
pos_ratio_INTJ                 16
dependency_ratio_discourse     15
dependency_ratio_vocative      19
pos_ratio_X                    19
dependency_ratio_list          19
dependency_ratio_dislocated    18
dependency_ratio_reparandum    19
dtype: int64

So it seems lots of ratios dividing by zero to get NaNs, so actually want to pre-define the tags. Will copy paste pipeline here again to make version 3

___________

# NOTE TO ME

Can begin rerunnig from here

In [16]:
bigrams = pd.read_csv("final_chosen_bigrams.csv")

In [17]:
chosen_bigrams_list = list(
    bigrams[["pos_1", "pos_2"]]
    .itertuples(index = False, name = None)
)

print(chosen_bigrams_list)

[('ADJ', 'NOUN'), ('NOUN', 'PUNCT'), ('DET', 'NOUN'), ('NOUN', 'ADP'), ('ADP', 'DET'), ('VERB', 'ADP'), ('DET', 'ADJ'), ('AUX', 'VERB'), ('VERB', 'DET'), ('ADP', 'NOUN'), ('PRON', 'VERB'), ('NOUN', 'AUX'), ('PART', 'VERB'), ('NOUN', 'VERB'), ('VERB', 'NOUN'), ('PRON', 'AUX'), ('ADV', 'VERB'), ('NOUN', 'NOUN'), ('NOUN', 'CCONJ'), ('PUNCT', 'CCONJ'), ('VERB', 'PUNCT'), ('ADP', 'ADJ'), ('VERB', 'PRON'), ('ADJ', 'PUNCT'), ('AUX', 'ADV'), ('PUNCT', 'PRON'), ('VERB', 'ADV'), ('ADP', 'PRON'), ('CCONJ', 'VERB'), ('PUNCT', 'VERB'), ('PRON', 'NOUN'), ('NOUN', 'ADV'), ('ADJ', 'ADP'), ('VERB', 'ADJ')]


In [18]:
def extract_syntactic_features_3(text):

    # run cleaning function
    text = syntactic_pipe_clean(text)
    # return empty dict if no text
    if not text:
        return {}

    # run text through stanza pipeline to parse
    text = stan_pipe(text)

    # get total sentences - this is used multiple times below
    num_sentences = len(text.sentences)

    
    ##################################
    # POS features
    ##################################

    # initialise counters for pos tags
    pos_counts = Counter()
    pos_bigrams = Counter()
    total_tokens = 0

    # loop through sentences in text
    for sentence in text.sentences:

        # create empty list for POS tags in current setnence
        sentence_pos_tags = []

        ##################################
        # POS tag counts
        ##################################

        # getting pos tags in current sentence
        # loop through words in sentence
        for word in sentence.words:

            # get POS tag for current word
            pos_tag = word.upos
            # increment total for that POS tag
            pos_counts[pos_tag] += 1
            # add current pos tag to list for this sentence
            sentence_pos_tags.append(pos_tag)
            # increment tokens processed
            total_tokens += 1

        ##################################
        # POS bigrams
        ##################################

        # now looking at bigrams
        # loop through all POS tags in current sentence (except last) 
        for i in range(len(sentence_pos_tags) - 1):
            # and take the bigram from each, that is, current tag + next
            bigram = (
                sentence_pos_tags[i],
                sentence_pos_tags[i + 1]
            )
            # and now check whether that bigram is in our list of chosen bigrams
            if bigram in chosen_bigrams_list:
                
                # ...if so, increment the freq of this bigram
                pos_bigrams[bigram] += 1

    # now get the totals
    total_bigrams = sum(pos_bigrams.values())
    # total_trigrams = sum(pos_trigrams.values()) - too many features, removed, delete later?
    

    
    ##################################
    # Constituency features
    ##################################

    # set counter for storing phrase counts
    phrase_counts = Counter()
    # and list for parse tree depths
    parse_tree_depths = []
    # and specify phrases to measue where i.e. NP is noun phrase etc.
    target_phrases = {"NP", "VP", "PP", "ADJP", "ADVP"}

    # loop through sentences in text
    for sentence in text.sentences:

        # convert stansa parse tree to nltk tree to use nltk methods
        constituency_tree = Tree.fromstring(
            str(sentence.constituency)
        )
        # get height of tree and add to list of depths
        parse_tree_depths.append(
            constituency_tree.height()
        )
        # now loop through subtrees in tree...
        for subtree in constituency_tree.subtrees():
            # ...and get its label (NP etc.)
            phrase_label = subtree.label()
            # if label is one of those above in target
            if phrase_label in target_phrases:
                # increment it
                phrase_counts[phrase_label] += 1

    # now get total
    mean_parse_tree_depth = (
        np.mean(parse_tree_depths)
        if parse_tree_depths
        else 0
    )

    
    ##################################
    # Clause features
    ##################################

    # set empty list to store clausal dependents in each sentence
    clausal_dep_per_sent = []
    # initialise counter for sentences with at least 1
    sents_with_clausal_dep = 0
    # define the relations that represent clausal dependent strauctures
    # i.e. csubj = clausal subject
    clausal_relations = {"acl", "advcl", "ccomp", "csubj", "xcomp"}

    # loop through sentences in text
    for sentence in text.sentences:

        # ...and count the words whose dependency relation is listed above
        dep_clause_count = sum(
            # initially returned too much info, here we now collapse sub-types
            word.deprel.split(":")[0] in clausal_relations
            for word in sentence.words
        )
        # add that number to list
        clausal_dep_per_sent.append(dep_clause_count)
        # confirm there is at least one
        if dep_clause_count > 0:
            # if so increment sentences that have a clausal dependent
            sents_with_clausal_dep += 1

    # now get totals
    mean_clausal_dep_per_sent = (
        np.mean(clausal_dep_per_sent)
        if clausal_dep_per_sent
        else 0
    )
    sent_with_clausal_dep_ratio = (
        sents_with_clausal_dep / num_sentences
        if num_sentences > 0
        else 0
    )

    
    ##################################
    # Production rule features
    ##################################

    # this created way too many features...
    # ...and whilst we don't want to omit this category, we now switch to 
    # mean production rules per sentence, solely to keep output manageable

    # initialise empty list
    prod_rules_per_sent = []

    # loop through sentences in text
    for sentence in text.sentences:
    
        # convert stanza parse to NLTK tree
        nltk_tree = Tree.fromstring(str(sentence.constituency))
        # set counter to 0 for this sentence
        sent_prod_rule_count = 0
        
        # and now get the production rules
        for production in nltk_tree.productions():
        
            # lexical rules retain words from the text on the right hand side
            # not only do we are measuring syntactic strusture and not vocabulary
            # but this results in hundreds of additional features
            if not production.is_lexical():

                # if we find a production rule, increment the count
                sent_prod_rule_count += 1

        # add the sentence prod rule count to the lsit
        prod_rules_per_sent.append(sent_prod_rule_count)

    # and finally calculate the mean
    mean_prod_rules_per_sent = (
        np.mean(prod_rules_per_sent)
        if prod_rules_per_sent
        else 0
    )

    
    ##################################
    # Dependency features
    ##################################

    # set counter to store freq of dependency relations
    dep_rel_counts = Counter()
    # set empty list to store ave. dependency depths
    dep_depths = []
    # set empty list to store distances between each word and head
    dep_distances = []
    
    # loop through sentences in text
    for sentence in text.sentences:

        # loop through words in sentence
        for word in sentence.words:

            # this initally resulted in massive numbers of features due to subtypes
            # so we want to collapse the relation to just the main relation
            collapsed_rel = (word.deprel.split(":")[0])
            # and for each increment count
            dep_rel_counts[collapsed_rel] += 1
        
            # now for distances...
            # exclude the root (0) because it cannot have a head 
            # and therefore no distance to the head
            if word.head != 0:

                # then get distance between word and head
                # abs to ensure positive
                dep_distance = abs(word.id - word.head)
                # store it
                dep_distances.append(dep_distance)

        
        # now for tree depth
        # loop through each word in the sentence and for each 
        # create an entry that is "that word id" : "word head id"
        paths = {
            word.id : word.head
            for word in sentence.words
        }
        # create empty list for dependency depths for each word
        sent_dep_depths = []

        # now loop through each word in sentence
        for word in sentence.words:
            # get depth of current word and store
            depth = get_dep_depth(word.id, paths)
            sent_dep_depths.append(depth)

        # finally get average dependency depth
        if sent_dep_depths:
            dep_depths.append(np.mean(sent_dep_depths))

    # now get totals
    tot_dependencies = sum(
        dep_rel_counts.values()
    )
    mean_dep_depth = (
        np.mean(dep_depths)
        if dep_depths
        else 0
    )
    mean_dep_distance = (
        np.mean(dep_distances)
        if dep_distances
        else 0
    )


    ##################################
    # Storage
    ##################################

    # now need to store all variables gathered in the pipeline
    # start a dict here with what we have already
    syntactic_features = {

        # constituency
        "mean_parse_tree_depth" :       mean_parse_tree_depth,

        # clausal dependent
        "mean_clausal_dep_per_sent" :   mean_clausal_dep_per_sent,
        "sent_with_clausal_dep_ratio" : sent_with_clausal_dep_ratio,
        
        # production rule
        "mean_prod_rules_per_sent" :   mean_prod_rules_per_sent,
        
        # dependency
        "mean_dep_depth" :              mean_dep_depth,
        "mean_dep_distance" :           mean_dep_distance
    
    }

    ##################################
    # Storage of additional features
    ##################################

    
    # define list of possible pos tags
    pos_tag_list = [
        "ADJ", "ADP", "ADV", "AUX", "CCONJ",
        "DET", "INTJ", "NOUN", "NUM", "PART",
        "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", 
        "VERB", "X"
    ]

    # POS tag ratios
    if total_tokens > 0:

        # loop through all possible pos tags 
        for pos_tag in pos_tag_list:

            # get count
            count = pos_counts[pos_tag]
            
            # and store ratio of count / total as "pos_ratio_tag"
            syntactic_features[f"pos_ratio_{pos_tag}"] = (
                count / total_tokens
            )

    # POS bigram frequencies
    # loop through list of chosen bigrams
    for bigram in chosen_bigrams_list:
        
        # and get the count of it, if doesn't exist return 0
        count = pos_bigrams.get(bigram, 0)

        # and store ratio of count / total as "pos_bigram_word1_word2"
        syntactic_features[f"pos_bigram_{bigram[0]}_{bigram[1]}"] = (
            count / total_bigrams
            if total_bigrams > 0
            else 0
        )
    
    # phrase type frequencies
    if num_sentences > 0:

        # loop through all phrase targets 
        for phrase in target_phrases:

            # get count
            count = phrase_counts[phrase]
            
            # and store ratio of count / total as "phrase_per_sent_"phrase""
            syntactic_features[f"phrase_per_sent_{phrase}"] = (
                count / num_sentences
            )

    # dependency relation frequencies

    dependency_relation_list = [
        "acl", "advcl", "advmod", "amod", "appos", "aux", "case", "cc",
        "ccomp", "clf", "compound", "conj", "cop", "csubj", "dep", "det",
        "discourse", "dislocated", "expl", "fixed", "flat", "iobj", "mark",
        "nmod", "nsubj", "nummod", "obj", "obl", "orphan", "parataxis", 
        "punct", "reparandum", "root", "vocative", "xcomp"
    ]
       
    if tot_dependencies > 0:

        # loop through all relation counts 
        for relation in dependency_relation_list:
            
            # get count
            count = dep_rel_counts[relation]
            # and store ratio of count / total as "dependency_ratio_'relation'"
            syntactic_features[f"dependency_ratio_{relation}"] = (
                count / tot_dependencies
            )
    
    # ...and return all features
    return syntactic_features

And now we try again...

In [25]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run syntactic feature extraction on first 20 texts to ensure it works as expected
X_train_syntactic = pd.DataFrame(
    train_df["text"].iloc[:20].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_syntactic.shape)
print(X_train_syntactic.isna().sum().sum())
print(X_train_syntactic.dtypes.value_counts())
X_train_syntactic.head(10)

(20, 97)
0
float64    97
Name: count, dtype: int64


,mean_parse_tree_depth,mean_clausal_dep_per_sent,sent_with_clausal_dep_ratio,mean_prod_rules_per_sent,mean_dep_depth,mean_dep_distance,pos_ratio_ADJ,pos_ratio_ADP,pos_ratio_ADV,pos_ratio_AUX,...,dependency_ratio_nummod,dependency_ratio_obj,dependency_ratio_obl,dependency_ratio_orphan,dependency_ratio_parataxis,dependency_ratio_punct,dependency_ratio_reparandum,dependency_ratio_root,dependency_ratio_vocative,dependency_ratio_xcomp
0,10.954545,1.045455,0.681818,16.863636,2.292963,3.173594,0.095128,0.095128,0.039443,0.034803,...,0.006961,0.039443,0.051044,0.0,0.000000,0.106729,0.0,0.051044,0.000000,0.013921
1,13.750000,1.687500,0.750000,23.687500,2.426383,3.772840,0.114014,0.073634,0.066508,0.078385,...,0.002375,0.047506,0.049881,0.0,0.002375,0.104513,0.0,0.038005,0.000000,0.011876
2,13.888889,1.555556,0.833333,21.555556,2.688738,3.200935,0.089686,0.109865,0.011211,0.053812,...,0.011211,0.047085,0.060538,0.0,0.000000,0.114350,0.0,0.040359,0.000000,0.011211
3,13.842105,2.052632,0.947368,22.105263,2.626988,3.576674,0.085062,0.093361,0.026971,0.062241,...,0.006224,0.033195,0.066390,0.0,0.002075,0.105809,0.0,0.039419,0.000000,0.016598
4,10.370370,1.111111,0.481481,15.666667,1.895112,3.309577,0.092437,0.084034,0.058824,0.046218,...,0.010504,0.063025,0.050420,0.0,0.006303,0.100840,0.0,0.056723,0.000000,0.029412
5,12.882353,1.235294,0.705882,22.529412,2.527591,3.378076,0.043103,0.142241,0.015086,0.015086,...,0.004310,0.045259,0.081897,0.0,0.000000,0.107759,0.0,0.036638,0.000000,0.004310
6,10.666667,1.428571,0.619048,21.380952,2.214483,3.738574,0.058099,0.089789,0.070423,0.044014,...,0.005282,0.040493,0.056338,0.0,0.008803,0.188380,0.0,0.036972,0.003521,0.012324
7,12.000000,1.894737,0.789474,20.842105,2.145926,3.502488,0.106888,0.076010,0.073634,0.066508,...,0.000000,0.047506,0.045131,0.0,0.002375,0.095012,0.0,0.045131,0.000000,0.026128
8,12.722222,1.222222,0.555556,22.333333,2.679917,4.056911,0.092157,0.064706,0.047059,0.039216,...,0.013725,0.047059,0.037255,0.0,0.001961,0.170588,0.0,0.035294,0.000000,0.005882
9,9.487179,0.948718,0.564103,12.512821,1.759073,2.877049,0.047438,0.087287,0.034156,0.039848,...,0.003795,0.047438,0.066414,0.0,0.005693,0.168880,0.0,0.074004,0.000000,0.017078


Much better. Can now re-run across the whole train set.

In [26]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\train_dataset_master.csv"
)

# run syntactic feature extraction across entire train set
X_train_syntactic = pd.DataFrame(
    train_df["text"].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_syntactic.shape)
print(X_train_syntactic.isna().sum().sum())
print(X_train_syntactic.dtypes.value_counts())
X_train_syntactic.head(10)

(3200, 97)
0
float64    97
Name: count, dtype: int64


,mean_parse_tree_depth,mean_clausal_dep_per_sent,sent_with_clausal_dep_ratio,mean_prod_rules_per_sent,mean_dep_depth,mean_dep_distance,pos_ratio_ADJ,pos_ratio_ADP,pos_ratio_ADV,pos_ratio_AUX,...,dependency_ratio_nummod,dependency_ratio_obj,dependency_ratio_obl,dependency_ratio_orphan,dependency_ratio_parataxis,dependency_ratio_punct,dependency_ratio_reparandum,dependency_ratio_root,dependency_ratio_vocative,dependency_ratio_xcomp
0,10.954545,1.045455,0.681818,16.863636,2.292963,3.173594,0.095128,0.095128,0.039443,0.034803,...,0.006961,0.039443,0.051044,0.0,0.000000,0.106729,0.0,0.051044,0.000000,0.013921
1,13.750000,1.687500,0.750000,23.687500,2.426383,3.772840,0.114014,0.073634,0.066508,0.078385,...,0.002375,0.047506,0.049881,0.0,0.002375,0.104513,0.0,0.038005,0.000000,0.011876
2,13.888889,1.555556,0.833333,21.555556,2.688738,3.200935,0.089686,0.109865,0.011211,0.053812,...,0.011211,0.047085,0.060538,0.0,0.000000,0.114350,0.0,0.040359,0.000000,0.011211
3,13.842105,2.052632,0.947368,22.105263,2.626988,3.576674,0.085062,0.093361,0.026971,0.062241,...,0.006224,0.033195,0.066390,0.0,0.002075,0.105809,0.0,0.039419,0.000000,0.016598
4,10.370370,1.111111,0.481481,15.666667,1.895112,3.309577,0.092437,0.084034,0.058824,0.046218,...,0.010504,0.063025,0.050420,0.0,0.006303,0.100840,0.0,0.056723,0.000000,0.029412
5,12.882353,1.235294,0.705882,22.529412,2.527591,3.378076,0.043103,0.142241,0.015086,0.015086,...,0.004310,0.045259,0.081897,0.0,0.000000,0.107759,0.0,0.036638,0.000000,0.004310
6,10.666667,1.428571,0.619048,21.380952,2.214483,3.738574,0.058099,0.089789,0.070423,0.044014,...,0.005282,0.040493,0.056338,0.0,0.008803,0.188380,0.0,0.036972,0.003521,0.012324
7,12.000000,1.894737,0.789474,20.842105,2.145926,3.502488,0.106888,0.076010,0.073634,0.066508,...,0.000000,0.047506,0.045131,0.0,0.002375,0.095012,0.0,0.045131,0.000000,0.026128
8,12.722222,1.222222,0.555556,22.333333,2.679917,4.056911,0.092157,0.064706,0.047059,0.039216,...,0.013725,0.047059,0.037255,0.0,0.001961,0.170588,0.0,0.035294,0.000000,0.005882
9,9.487179,0.948718,0.564103,12.512821,1.759073,2.877049,0.047438,0.087287,0.034156,0.039848,...,0.003795,0.047438,0.066414,0.0,0.005693,0.168880,0.0,0.074004,0.000000,0.017078


In [27]:
# took half a day to run so save first
X_train_syntactic.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic.csv",
    index = False
)

See a few columns with zeroes. Checking to see if entire columns are zeroes.

In [29]:
zero_columns = X_train_syntactic.columns[(X_train_syntactic == 0).all()]

print("Number of zero columns: ", len(zero_columns))
print(zero_columns.to_list())

Number of zero columns:  1
['dependency_ratio_clf']


One column dependency_ratio_clf has 0s through all 3200 texts so we can drop as offers no value.

In [30]:
X_train_syntactic = X_train_syntactic.drop(columns = zero_columns)

print(X_train_syntactic.shape)

(3200, 96)


In [31]:
# overwrite older save
X_train_syntactic.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic.csv",
    index = False
)

___

# Returning for Objective 5

Had to re-run a 300 test set through paraphrasing and so now have a slightly reduced train set of 2900 that must have features extracted again so that training doesn't train on features that are now in the test set. New train set is:

C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv

Will now re-run on the new train set.

In [8]:
# load training data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\final_train_2900.csv"
)

# run syntactic feature extraction across entire train set
X_train_syntactic_2900 = pd.DataFrame(
    train_df["text"].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)

# check output
print(X_train_syntactic_2900.shape)
print(X_train_syntactic_2900.isna().sum().sum())
print(X_train_syntactic_2900.dtypes.value_counts())
X_train_syntactic_2900.head(10)

X_train_syntactic_2900.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic_2900.csv",
    index = False
)

(2899, 97)
0
float64    97
Name: count, dtype: int64


In [9]:
# final check to see if any columns are entirely zeroes
zero_columns = X_train_syntactic_2900.columns[(X_train_syntactic_2900 == 0).all()]

print("Number of zero columns: ", len(zero_columns))
print(zero_columns.to_list())

Number of zero columns:  1
['dependency_ratio_clf']


One column entirely zero as previous, lets remove and re-save.

In [10]:
X_train_syntactic_2900 = X_train_syntactic_2900.drop(columns = zero_columns)

print(X_train_syntactic_2900.shape)

(2899, 96)


In [11]:
X_train_syntactic_2900.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\X_train_syntactic_2900.csv",
    index = False
)

____

Now returning for the test data...

In [19]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300.csv"
)

# run syntactic feature extraction across entire test set
test_heavy_300 = pd.DataFrame(
    train_df["text"].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)
# drop the column that has repeatedly been only zeroes
test_heavy_300 = test_heavy_300.drop(columns = "dependency_ratio_clf")

# check output
print(test_heavy_300.shape)
print(test_heavy_300.isna().sum().sum())
print(test_heavy_300.dtypes.value_counts())
test_heavy_300.head(10)

test_heavy_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_heavy_300_features_syntactic.csv",
    index = False
)

(300, 96)
0
float64    96
Name: count, dtype: int64


In [20]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300.csv"
)

# run syntactic feature extraction across entire test set
test_light_300 = pd.DataFrame(
    train_df["text"].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)
# drop the column that has repeatedly been only zeroes
test_light_300 = test_light_300.drop(columns = "dependency_ratio_clf")

# check output
print(test_light_300.shape)
print(test_light_300.isna().sum().sum())
print(test_light_300.dtypes.value_counts())
test_light_300.head(10)

test_light_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_light_300_features_syntactic.csv",
    index = False
)

(300, 96)
0
float64    96
Name: count, dtype: int64


In [21]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300.csv"
)

# run syntactic feature extraction across entire test set
test_original_300 = pd.DataFrame(
    train_df["text"].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)
# drop the column that has repeatedly been only zeroes
test_original_300 = test_original_300.drop(columns = "dependency_ratio_clf")

# check output
print(test_original_300.shape)
print(test_original_300.isna().sum().sum())
print(test_original_300.dtypes.value_counts())
test_original_300.head(10)

test_original_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_original_300_features_syntactic.csv",
    index = False
)

(300, 96)
0
float64    96
Name: count, dtype: int64


In [22]:
# load test data
train_df = pd.read_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300.csv"
)

# run syntactic feature extraction across entire test set
test_trans_300 = pd.DataFrame(
    train_df["text"].apply(extract_syntactic_features_3
    # and convert to list of dicts
    ).to_list()
)
# drop the column that has repeatedly been only zeroes
test_trans_300 = test_trans_300.drop(columns = "dependency_ratio_clf")

# check output
print(test_trans_300.shape)
print(test_trans_300.isna().sum().sum())
print(test_trans_300.dtypes.value_counts())
test_trans_300.head(10)

test_trans_300.to_csv(
    r"C:\Users\Jamie\Desktop\DSM500 Final Project\Code\Final datasets v2 last update\test_trans_300_features_syntactic.csv",
    index = False
)

(300, 96)
0
float64    96
Name: count, dtype: int64


Now done.